<a href="https://colab.research.google.com/github/jasonmcgovern/gallery/blob/main/hubSpotImpersonator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
import pandas as pd
import openai
from openai import OpenAI
import time
from collections import Counter

In [ ]:
# @title
# === CONFIGURATION ===
USE_GPT = True  # Set to True to use GPT, False for rule-based
OPENAI_API_KEY = "[input open API key here]"
INPUT_CSV = "keywords.csv"
OUTPUT_CSV = "keywords_rated.csv"

In [ ]:
# @title Initialize Functions
# === OpenAI Setup ===
import time
from datetime import datetime
import json

if USE_GPT:
    client = openai.OpenAI(api_key=OPENAI_API_KEY)

def getPrompts(keyword):
    prompt = f"""Generate exactly 100 realistic search queries for the category: {keyword}.

GOAL: Create "Inquisitive" queries from a user who is in the research phase and does NOT mention specific brands.

INSTRUCTIONS:
1. NO BRANDS: Do not include any specific company names, trademarks, or brand identifiers.
2. DIVERSE PERSONAS: Generate queries from 4 different perspectives:
   - The Beginner: (e.g., "What should I look for when buying my first {keyword}?")
   - The Professional: (e.g., "High-performance {keyword} specs for daily commercial use.")
   - The Budget-Conscious: (e.g., "Most durable {keyword} options under $500.")
   - The Problem-Solver: (e.g., "Which type of {keyword} is best for heavy-duty tasks?")
3. TECHNICAL FOCUS: Encourage queries about specs, power, materials, or service levels instead of names.

FORMAT: Return ONLY a valid JSON array of strings.
Example: ["query 1", "query 2", ...]"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that returns responses in valid JSON format only. Never use markdown code blocks."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        response_text = response.choices[0].message.content.strip()

        # Remove markdown code blocks if present
        if response_text.startswith("```"):
            response_text = response_text.split("```")[1]
            if response_text.startswith("json"):
                response_text = response_text[4:]
            response_text = response_text.strip()

        # Debug: print what we got
        print(f"Raw response length: {len(response_text)}")
        print(f"First 200 chars: {response_text[:200]}")

        prompts = json.loads(response_text)

        # Validate it's actually a list
        if not isinstance(prompts, list):
            print(f"Warning: Expected list but got {type(prompts)}")
            return []

        print(f"Successfully parsed {len(prompts)} prompts")
        return prompts

    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text}")
        return []
    except Exception as e:
        print(f"Error: {e}")
        return []

def runPrompts(prompts):
    """
    Run a list of prompts through ChatGPT and collect responses

    Args:
        prompts: List of prompt strings

    Returns:
        List of dictionaries with prompt, response, and metadata
    """
    results = []

    print(f"Starting to process {len(prompts)} prompts...")

    for i, prompt in enumerate(prompts):
        # Progress indicator
        print(f"Processing {i+1}/{len(prompts)}: {prompt[:60]}{'...' if len(prompt) > 60 else ''}")

        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=500  # Limit response length for cost control
            )

            result = {
                'prompt_number': i + 1,
                'prompt': prompt,
                'response': response.choices[0].message.content.strip(),
                'timestamp': datetime.now().isoformat(),
                'tokens_used': response.usage.total_tokens if hasattr(response, 'usage') else None
            }

            results.append(result)
            print(f"✓ Success - {len(result['response'])} characters received")

            # Small delay to avoid rate limits (adjust as needed)
            time.sleep(0.5)

        except Exception as e:
            print(f"✗ Error with prompt {i+1}: {str(e)}")

            # Still add an entry so we don't lose track
            error_result = {
                'prompt_number': i + 1,
                'prompt': prompt,
                'response': None,
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            }
            results.append(error_result)

            # Continue with next prompt
            continue

    successful = len([r for r in results if r.get('response') is not None])
    print(f"\nCompleted! {successful}/{len(prompts)} prompts successful")

    return results

def extractEntitiesWithGPT(responses, keyword):
    """
    Extract brand/company entities from responses using GPT

    Args:
        responses: List of response dictionaries from runPrompts
        keyword: The keyword used to generate prompts (e.g. "golf balls", "CRM software")

    Returns:
        Tuple of (updated_responses, entity_summary)
    """
    print(f"Extracting {keyword} brands from {len(responses)} responses...")

    all_entities = []
    successful_extractions = 0

    for i, result in enumerate(responses):
        # Skip if there was an error in the original response
        if result.get('response') is None:
            result['entities'] = []
            continue

        print(f"Extracting entities {i+1}/{len(responses)}...")

        extraction_prompt = f"""Analyze this text and extract ONLY actual brand names and company names that sell {keyword}.

Text to analyze: "{result['response']}"

IMPORTANT RULES:
- ONLY include proper nouns (capitalized names) of actual companies/brands
- DO NOT include product types like "fixed annuities", "variable annuities", "indexed annuities"
- DO NOT include generic descriptive terms
- DO NOT include features or categories
- ONLY include real company names (e.g., "Fidelity", "Vanguard", "Northwestern Mutual", "MetLife")
- DO NOT include ANY placeholder references like "Brand A", "Brand B", "Company X", "Product 1", etc.

Examples of what TO include:
- Titleist (golf ball brand)
- Salesforce (CRM software brand)
- Prudential (insurance company)
- Lakeside Family Dental (practice name)
- Summit Orthodontics (practice name)

Examples of what NOT to include:
- "fixed annuities" (product type, not a brand)
- "tax-deferred" (feature)
- "immediate annuities" (product category)
- "Brand A" or "Brand B" (placeholders)
- "Company X" (placeholder)

Return as a simple JSON array of strings: ["Brand1", "Brand2", "Brand3"]
If no actual brand/company names are mentioned, return: []

Response:"""

        try:
            extraction_response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are an expert at identifying actual brand and company names. You distinguish between product categories and brand names. You NEVER include placeholder names like 'Brand A' or 'Company X'. Return only valid JSON arrays."},
                    {"role": "user", "content": extraction_prompt}
                ],
                temperature=0,  # Low temperature for consistent extraction
                max_tokens=200
            )

            # Parse the JSON response
            entities_text = extraction_response.choices[0].message.content.strip()

            # Remove markdown if present
            if entities_text.startswith("```"):
                entities_text = entities_text.split("```")[1]
                if entities_text.startswith("json"):
                    entities_text = entities_text[4:]
                entities_text = entities_text.strip()

            entities = json.loads(entities_text)

            # Additional filtering: remove common false positives
            false_positives = [
                'fixed annuities', 'variable annuities', 'indexed annuities',
                'immediate annuities', 'deferred annuities', 'tax-deferred',
                'fixed', 'variable', 'indexed', 'immediate', 'deferred',
                'annuity', 'annuities', 'insurance', 'investment'
            ]

            # Comprehensive placeholder patterns to filter out
            import re

            def is_placeholder(entity):
                """Check if entity is a placeholder"""
                entity_lower = entity.lower().strip()

                # Direct matches for common placeholders
                placeholder_list = [
                    'brand a', 'brand b', 'brand c', 'brand x', 'brand y', 'brand z',
                    'branda', 'brandb', 'brandc', 'brandx', 'brandy', 'brandz',
                    'company a', 'company b', 'company c', 'company x', 'company y', 'company z',
                    'companya', 'companyb', 'companyc', 'companyx', 'companyy', 'companyz',
                    'product a', 'product b', 'product c', 'product x', 'product y', 'product z',
                    'producta', 'productb', 'productc', 'productx', 'producty', 'productz',
                    'option a', 'option b', 'option c', 'option 1', 'option 2', 'option 3',
                    'provider a', 'provider b', 'provider c', 'provider x', 'provider y',
                    'firm a', 'firm b', 'firm c', 'firm x', 'firm y',
                    'insurer a', 'insurer b', 'insurer c', 'insurer x', 'insurer y'
                ]

                if entity_lower in placeholder_list:
                    return True

                # Pattern matching for variations
                placeholder_patterns = [
                    r'^brand\s*[a-z]$',           # "Brand A", "Brand B", etc.
                    r'^company\s*[a-z]$',         # "Company A", "Company B", etc.
                    r'^product\s*[a-z0-9]$',      # "Product A", "Product 1", etc.
                    r'^option\s*[a-z0-9]$',       # "Option A", "Option 1", etc.
                    r'^provider\s*[a-z]$',        # "Provider A", "Provider B", etc.
                    r'^firm\s*[a-z]$',            # "Firm A", "Firm B", etc.
                    r'^insurer\s*[a-z]$',         # "Insurer A", "Insurer B", etc.
                    r'^brand[a-z]$',              # "BrandA", "BrandB", etc. (no space)
                    r'^company[a-z]$',            # "CompanyA", "CompanyB", etc. (no space)
                ]

                for pattern in placeholder_patterns:
                    if re.match(pattern, entity_lower):
                        return True

                return False

            # Filter out false positives and placeholders
            entities = [
                e for e in entities
                if e.lower() not in false_positives and not is_placeholder(e)
            ]

            # Store entities in the result
            result['entities'] = entities
            all_entities.extend(entities)
            successful_extractions += 1

            print(f"✓ Found {len(entities)} entities: {entities}")

            # Rate limiting
            time.sleep(0.3)

        except Exception as e:
            print(f"✗ Error extracting entities from response {i+1}: {str(e)}")
            result['entities'] = []
            continue

    # Create entity summary
    from collections import Counter
    entity_counts = Counter(all_entities)

    entity_summary = {
        'total_responses_processed': len(responses),
        'successful_extractions': successful_extractions,
        'unique_entities': len(entity_counts),
        'total_mentions': len(all_entities),
        'entity_counts': dict(entity_counts),
        'top_entities': entity_counts.most_common(10),
        'keyword': keyword
    }

    print(f"\nEntity Extraction Complete!")
    print(f"✓ {successful_extractions}/{len(responses)} responses processed")
    print(f"✓ Found {len(entity_counts)} unique entities")
    print(f"✓ Total mentions: {len(all_entities)}")
    print(f"Top 5 entities: {entity_counts.most_common(5)}")

    return responses, entity_summary

In [ ]:
# @title
# === MAIN EXECUTION ===

# Step 1: Generate prompts
keyword = "orthodontists in denver colorado"
print(f"Generating prompts for: {keyword}")
prompts = getPrompts(keyword)
print(f"Generated {len(prompts)} prompts")

# Step 2: Run prompts through ChatGPT
print(f"\nRunning prompts through ChatGPT...")
responses = runPrompts(prompts)

# Step 3: Extract entities from responses
print(f"\nExtracting brand entities...")
responses_with_entities, entity_summary = extractEntitiesWithGPT(responses, keyword)

# Step 4: Display results
print(f"\n=== BRAND ANALYSIS RESULTS ===")
print(f"Keyword: {entity_summary['keyword']}")
print(f"Total queries processed: {entity_summary['total_responses_processed']}")
print(f"Unique brands found: {entity_summary['unique_entities']}")
print(f"Total brand mentions: {entity_summary['total_mentions']}")

print(f"\n=== SHARE OF VOICE ===")
for brand, count in entity_summary['top_entities']:
    percentage = (count / entity_summary['total_mentions']) * 100 if entity_summary['total_mentions'] > 0 else 0
    print(f"{brand}: {count} mentions ({percentage:.1f}%)")

# Step 5: Show sample responses
print(f"\n=== SAMPLE RESPONSES ===")
for i, result in enumerate(responses_with_entities[:3]):
    if result.get('response'):
        print(f"\nQuery {i+1}: {result['prompt']}")
        print(f"Response: {result['response'][:200]}...")
        print(f"Brands mentioned: {result.get('entities', [])}")
        print("-" * 50)